### 目录

1、数据库概览方式

2、数据清洗的主要目标及流程

3、数据清洗方式

3.1 处理不一致值

3.2 处理缺失值

3.3 处理异常值

3.4 处理不合理值及重复值

4、数据类别

5、特征构造与编码

In [1]:
import pandas as pd
import numpy as np

# 严格按照菜鸟教程要求，构造【含5类数据问题】的原始脏数据
df = pd.DataFrame({
    # 用户ID（正常主键）
    "用户ID": [1001, 1002, 1003, 1004, 1002, 1005, 1006, 1007, 1008, 1009],
    # 姓名：含缺失值
    "姓名": ["张三", "李四", "王五", np.nan, "李四", "钱七", "孙八", "周九", "吴十", np.nan],
    # 性别：不一致值（male/M/女/F/female）+ 混合格式
    "性别": ["男", "male", "M", "女", "male", "F", "female", "男", "F", "女"],
    # 年龄：缺失值 + 错误值(负数) + 异常值(150)
    "年龄": [25, 30, -6, 45, 30, 150, 28, np.nan, 32, 40],
    # 城市：不一致值(大小写混合) + 缺失值
    "城市": ["beijing", "Shanghai", "guangzhou", "shenzhen", "Shanghai", np.nan, "Beijing", "hangzhou", "Nanjing", "shenzhen"],
    # 年收入：异常值(极端大) + 错误值(负数)
    "年收入": [80000, 120000, 75000, 95000, 120000, -5000, 110000, 98000, 300000, 105000],
    # 上次消费金额：缺失值
    "上次消费金额": [200, 500, np.nan, 300, 500, 150, np.nan, 400, 250, 350]
})

# 导出为CSV文件（和.ipynb同目录，无索引，纯原始脏数据）
df.to_csv("customer_data.csv", index=False, encoding="utf-8-sig")

### 1、数据库概览方式

数据集形状、前五行、基本信息、统计描述

检查空值、列名、列统计

In [3]:
import pandas as pd
import numpy as np

# 加载数据集
df = pd.read_csv('customer_data.csv') # 请替换为你的文件路径

# 查看数据的基本信息和前几行
print("数据集形状（行，列）:", df.shape)
print("\n数据前5行：", df.head())
print("\n数据基本信息：", df.info())
print("\n数据统计描述：", df.describe())
print("\n每列空值数量", df.isnull().sum())
print("数据集所有列名：", df.columns)

# 统计某一列值出现
df["姓名"].value_counts()

数据集形状（行，列）: (10, 7)

数据前5行：    用户ID   姓名    性别    年龄         城市     年收入  上次消费金额
0  1001   张三     男  25.0    beijing   80000   200.0
1  1002   李四  male  30.0   Shanghai  120000   500.0
2  1003   王五     M  -6.0  guangzhou   75000     NaN
3  1004  NaN     女  45.0   shenzhen   95000   300.0
4  1002   李四  male  30.0   Shanghai  120000   500.0
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   用户ID    10 non-null     int64  
 1   姓名      8 non-null      str    
 2   性别      10 non-null     str    
 3   年龄      9 non-null      float64
 4   城市      9 non-null      str    
 5   年收入     10 non-null     int64  
 6   上次消费金额  8 non-null      float64
dtypes: float64(2), int64(2), str(3)
memory usage: 692.0 bytes

数据基本信息： None

数据统计描述：               用户ID          年龄            年收入     上次消费金额
count    10.000000    9.000000      10.000000    8.00000
mean   1004.700000   41.555556  10980

姓名
李四    2
张三    1
王五    1
钱七    1
孙八    1
周九    1
吴十    1
Name: count, dtype: int64


### 2、数据清洗的主要目标

流程：不一致值、缺失值、异常值、不合理值、重复值

In [1]:
import pandas as pd

# 构建数据质量维度数据表
data_quality = pd.DataFrame({
    "目标(英文)": ["完整性(Completeness)", "一致性(Consistency)", "准确性(Accuracy)", "唯一性(Uniqueness)", "合理性(Validity)"],
    "对象": ["缺失值", "不一致值", "异常值", "重复值", "错误值"],
    "描述": [
        "数据记录中某些字段的值为空（NaN, NULL）",
        "数据格式、单位或编码不统一（如\"男\"、\"Male\"、\"M\"）",
        "与大多数数据明显偏离的极端值，扭曲统计结果，识别（IQR、Z-Score）后删除或修正",
        "数据集中存在完全相同的记录，使模型过度偏向重复样本，影响泛化能力",
        "确保数据符合逻辑或业务规则（如年龄不能为负）"
    ]
})

# Jupyter 中直接显示美观表格
data_quality#.style.set_properties(**{'text-align': 'left'})

,目标(英文),对象,描述
0,完整性(Completeness),缺失值,"数据记录中某些字段的值为空（NaN, NULL）"
1,一致性(Consistency),不一致值,"数据格式、单位或编码不统一（如""男""、""Male""、""M""）"
2,准确性(Accuracy),异常值,与大多数数据明显偏离的极端值，扭曲统计结果，识别（IQR、Z-Score）后删除或修正
3,唯一性(Uniqueness),重复值,数据集中存在完全相同的记录，使模型过度偏向重复样本，影响泛化能力
4,合理性(Validity),错误值,确保数据符合逻辑或业务规则（如年龄不能为负）


### 3、数据清洗方式

#### 3.1 不一致值：映射统一编码+统一数据类型

In [ ]:
# 步骤1：去除字符串前后空格
df['性别'] = df['性别'].str.strip()

# 步骤2：映射统一编码（核心方法）
gender_map = {'男': '男', 'Male': '男', 'M': '男', '女': '女', 'F': '女'}
df['性别'] = df['性别'].replace(gender_map)
# 也可以使用 .map() 函数，但 .replace() 更灵活，未指定的值保持不变

# 步骤3：统一日期格式
df['注册日期'] = pd.to_datetime(df['注册日期'])

# 步骤4：转换数据类型：确保数值列是数字类型
if '年龄' in df.columns:
    df['年龄'] = pd.to_numeric(df['年龄'], errors='coerce') # errors='coerce'将错误转换为NaN

#### 3.2 处理缺失值

缺失数据的类型

（1）随机丢失（MAR，Missing at Random）

随机丢失意味着数据丢失的概率与丢失的数据本身无关，而仅与部分已观测到的数据有关。
也就是说，数据的缺失不是完全随机的，该类数据的缺失依赖于其他完全变量。

（2）完全随机丢失（MCAR，Missing Completely at Random）

数据的缺失是完全随机的，不依赖于任何不完全变量或完全变量，**不影响样本的无偏性**。
简单来说，就是数据丢失的概率与其假设值以及其他变量值都完全无关。

（3）非随机丢失（MNAR，Missing not at Random）

数据的缺失与不完全变量自身的取值有关。
分为两种情况：**缺失值取决于其假设值**（例如，高收入人群通常不希望在调查中透露他们的收入）；
或者，**缺失值取决于其他变量值**（假设女性通常不想透露她们的年龄，则这里年龄变量缺失值受性别变量的影响）。

![数据清洗流程图](1.jpg)

#### 3.2.1 删除

In [ ]:
# 方法一：删除
# 删除任何包含缺失值的行（适用于缺失值很少的情况）
df_dropped = df.dropna()

#### 3.2.2 填充/插补

#### 平均值填充（Mean/Mode Completer）
------------------------------------------------------------------
将初始数据集中的属性分为数值属性和非数值属性来分别进行处理。

如果空值是**数值型**的，就根据该属性在其他所有对象的取值的**平均值**来填充该缺失的属性值； 如果空值是**非数值型**的，就根据统计学中的**众数**原理，用该属性在其他所有对象的取值次数最多的值(即出现频率最高的值)来补齐该缺失的属性值。

与其相似的另一种方法叫**条件平均值填充法**（Conditional Mean Completer）。在该方法中，用于求平均的值并不是从数据集的所有对象中取，而是从与该对象具有**相同决策属性值**的对象中取得。

#### 热卡填充（Hot deck imputation，或就近补齐）
-------------------------------------------------------------------

对于一个包含空值的对象，热卡填充法在完整数据中找到一个与它**最相似的对象**，然后用这个**相似对象的值来进行填充**。不同的问题可能会选用不同的标准来对相似进行判定。该方法概念上很简单，且利用了数据间的关系来进行空值估计。这个方法的缺点在于难以定义相似标准，**主观因素较多**。

#### K最近距离邻法（K-means clustering）
--------------------------------------------------------------------

先根据欧式距离或相关分析来确定距离具有缺失数据样本最近的K个样本，将这K个值加权平均来估计该样本的缺失数据。

根据数据类型的不同，距离度量也不尽相同：
1、连续数据：最常用的距离度量有欧氏距离，曼哈顿距离以及余弦距离。
2、分类数据：汉明（Hamming）距离在这种情况比较常用。对于所有分类属性的取值，如果两个数据点的值不同，则距离加一。汉明距离实际上与属性间不同取值的数量一致。
KNN算法的一个明显缺点是，在分析大型数据集时会变得非常耗时，因为它会在整个数据集中搜索相似数据点。此外，在高维数据集中，最近与最远邻居之间的差别非常小，因此KNN的准确性会降低。